# LeakLens walkthrough: three leaks, one honest score

A model that scores suspiciously high is often leaking. In this notebook we plant three leaks,
watch them inflate the score, let LeakLens find them, fix each one and compare the honest result.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)

n = 2000
age = rng.integers(18, 80, n)
visits = rng.poisson(3, n)
bmi = rng.normal(27, 5, n).round(1)

# The real signal: older patients with more visits are a bit more likely to be readmitted
z = 0.04 * (age - 50) + 0.3 * (visits - 3) + 0.08 * (bmi - 27) - 0.5
readmitted = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

df = pd.DataFrame({"age": age, "visits": visits, "bmi": bmi, "readmitted": readmitted})
df.head()

,age,visits,bmi,readmitted
0,23,1,27.0,0
1,65,1,19.4,0
2,58,5,20.5,0
3,45,5,23.1,1
4,44,2,16.3,0


## 1. Plant three leaks

1. **Target leakage**: `discharge_code` is filled in *after* we know whether the patient came back, so it encodes the answer.
2. **Train/test overlap**: half of the test rows are copied into the training set.
3. **Scaler fitted before the split**: the scaler sees the test rows when it computes its mean and standard deviation.


In [2]:
from sklearn.preprocessing import StandardScaler

leaky = df.copy()

# Leak 1: a column that is only known after the outcome
leaky["discharge_code"] = leaky["readmitted"] * 2 + rng.integers(0, 2, n)
features = ["age", "visits", "bmi", "discharge_code"]

# Leak 3: the scaler is fitted on every row, test rows included
leaky[features] = StandardScaler().fit_transform(leaky[features])

train, test = train_test_split(leaky, test_size=0.25, random_state=SEED)

# Leak 2: half of the test rows sneak into the training set
train = pd.concat([train, test.sample(frac=0.5, random_state=SEED)])
len(train), len(test)

(1750, 500)

## 2. The inflated score

In [3]:
model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(train[features], train["readmitted"])
leaky_auc = roc_auc_score(test["readmitted"], model.predict_proba(test[features])[:, 1])
print(f"Leaky ROC AUC: {leaky_auc:.3f}")

Leaky ROC AUC: 1.000


## 3. Let LeakLens find the leaks

LeakLens audits the **data** (target leakage, train/test overlap) and the **code** (preprocessing before the split).
To keep this notebook from flagging itself, the leaky preprocessing step is copied into a string and scanned from there.

In [5]:
from leaklens.leakage import scan, split_overlap, score
from leaklens.scanner import scan_source

# Data audit
data_findings = scan(train, "readmitted") + split_overlap(train, test)

# Code audit: the same leaky step from cell 4, written as a plain script
LEAKY_CODE = """
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
"""
code_findings = scan_source(LEAKY_CODE, label="leaky_snippet")

findings = data_findings + code_findings
print(f"LeakLens trust score: {score(findings)}/100")
for f in findings:
    print(f"[{f.severity.upper()}] {f.kind} :: {f.column} :: {f.detail}")

LeakLens trust score: 0/100
[HIGH] target_leakage :: discharge_code :: alone predicts target (roc_auc=1.000)
[HIGH] split_overlap :: * :: 251 test rows (50.2%) also appear in train
[HIGH] preprocess_before_split :: leaky_snippet:5 :: StandardScaler.fit_transform() runs before the split at leaky_snippet:6


## 4. Fix each leak

1. **Target leakage**: drop `discharge_code`, it is not available at prediction time.
2. **Train/test overlap**: drop duplicate rows *before* splitting, and never copy test rows into train.
3. **Scaler before the split**: split first, then fit the scaler on the training rows only.

In [6]:
# Fix 2: start from the clean data and remove exact duplicates before splitting
clean = df.drop_duplicates()

# Fix 1: keep only the features we would really have when making a prediction
features = ["age", "visits", "bmi"]

train, test = train_test_split(clean, test_size=0.25, random_state=SEED)

# Fix 3: the scaler learns from the training rows only, then transforms both splits
scaler = StandardScaler().fit(train[features])
train, test = train.copy(), test.copy()
train[features] = scaler.transform(train[features])
test[features] = scaler.transform(test[features])
len(train), len(test)

(1493, 498)

## 5. Re-run LeakLens and get the honest score

In [7]:
FIXED_CODE = """
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
"""

findings = (
    scan(train, "readmitted")
    + split_overlap(train, test)
    + scan_source(FIXED_CODE, label="fixed_snippet")
)
print(f"LeakLens trust score: {score(findings)}/100")
for f in findings:
    print(f"[{f.severity.upper()}] {f.kind} :: {f.column} :: {f.detail}")

model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(train[features], train["readmitted"])
honest_auc = roc_auc_score(test["readmitted"], model.predict_proba(test[features])[:, 1])

print(f"\nLeaky ROC AUC:  {leaky_auc:.3f}")
print(f"Honest ROC AUC: {honest_auc:.3f}")

LeakLens trust score: 100/100

Leaky ROC AUC:  1.000
Honest ROC AUC: 0.669


## Takeaway

The leaky model looked perfect (ROC AUC 1.000) and LeakLens gave it a trust score of 0/100.
After fixing the three leaks, the trust score is 100/100 and the honest ROC AUC is about 0.67.
A lower score you can trust beats a perfect score that falls apart on new data.

Try it on your own data: `leaklens train.csv --target label --test test.csv`, or `leaklens --scan notebooks/` for code.